In [80]:
# ============================================================
# NOTEBOOK 01: EXPLORATORY DATA ANALYSIS & PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [81]:
# Load data
train = pd.read_csv('../data/raw/train.csv')
store = pd.read_csv('../data/raw/store.csv')

print("✅ All datasets loaded successfully!")
print(train.info)
print(store.info)

C:\Users\USER\AppData\Local\Temp\ipykernel_7656\1031884962.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/train.csv')


✅ All datasets loaded successfully!
<bound method DataFrame.info of          Store  DayOfWeek        Date  Sales  Customers  Open  Promo  \
0            1          5  2015-07-31   5263        555     1      1   
1            2          5  2015-07-31   6064        625     1      1   
2            3          5  2015-07-31   8314        821     1      1   
3            4          5  2015-07-31  13995       1498     1      1   
4            5          5  2015-07-31   4822        559     1      1   
...        ...        ...         ...    ...        ...   ...    ...   
1017204   1111          2  2013-01-01      0          0     0      0   
1017205   1112          2  2013-01-01      0          0     0      0   
1017206   1113          2  2013-01-01      0          0     0      0   
1017207   1114          2  2013-01-01      0          0     0      0   
1017208   1115          2  2013-01-01      0          0     0      0   

        StateHoliday  SchoolHoliday  
0                  0         

In [57]:
# Merge datasets
df = train.merge(store, on='Store', how='left')
print("✅ Successfully merged!")

✅ Successfully merged!


In [82]:
# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort values
df = df.sort_values('Date')

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print('✅ Successful')

Dataset shape: (844392, 30)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
✅ Successful


In [83]:
# Quick check
print(df.head())
print(df.info())

         Store  DayOfWeek       Date  Sales  Customers  Open  Promo  \
1016447    353          1 2013-01-01   3139        820     1      0   
1016827    733          1 2013-01-01  10765       2377     1      0   
1016656    562          1 2013-01-01   8498       1675     1      0   
1016770    676          1 2013-01-01   3821        777     1      0   
1016776    682          1 2013-01-01   3375        566     1      0   

         SchoolHoliday  CompetitionDistance  CompetitionOpenSinceMonth  ...  \
1016447              1                900.0                        0.0  ...   
1016827              1                860.0                       10.0  ...   
1016656              1               1210.0                        0.0  ...   
1016770              1               1410.0                        9.0  ...   
1016776              1                150.0                        9.0  ...   

         IsPromo  StoreType_b  StoreType_c  StoreType_d Assortment_b  \
1016447        0         T

In [61]:
# Remove days when store was closed
df = df[df['Open'] == 1]

In [62]:
# Fill missing competition distance with median (SAFE WAY)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(
    df['CompetitionDistance'].median()
)

# Fill remaining NaNs
df = df.fillna(0)

# Confirm
df.isnull().sum()

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64

In [63]:
# Extract time-based features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [64]:
# Create weekend indicator
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

In [65]:
# Rename for clarity
df['IsPromo'] = df['Promo']

In [66]:
# Convert categorical columns into numeric
df = pd.get_dummies(
    df,
    columns=['StoreType', 'Assortment', 'StateHoliday'],
    drop_first=True
)

In [67]:
# Split data based on time (VERY IMPORTANT)
train_df = df[df['Date'] < '2015-01-01']
test_df = df[df['Date'] >= '2015-01-01']

In [68]:
# Define features and target
X_train = train_df.drop(['Sales', 'Date'], axis=1)
y_train = train_df['Sales']

X_test = test_df.drop(['Sales', 'Date'], axis=1)
y_test = test_df['Sales']

# Check shapes
print(X_train.shape, X_test.shape)

(648360, 28) (196032, 28)


In [69]:
# quick sanity check
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print("\nSample Features:")
print(X_train.head())

Training Data: (648360, 28)
Testing Data: (196032, 28)

Sample Features:
         Store  DayOfWeek  Customers  Open  Promo  SchoolHoliday  \
1016447    353          1        820     1      0              1   
1016429    335          1        482     1      0              1   
1016606    512          1        625     1      0              1   
1016588    494          1        527     1      0              1   
1016624    530          1        532     1      0              1   

         CompetitionDistance  CompetitionOpenSinceMonth  \
1016447                900.0                        0.0   
1016429                 90.0                        0.0   
1016606                590.0                        0.0   
1016588               1260.0                        6.0   
1016624              18160.0                        0.0   

         CompetitionOpenSinceYear  Promo2  ...  IsPromo  StoreType_b  \
1016447                       0.0       1  ...        0         True   
1016429            

In [70]:
# Convert boolean columns to int
bool_cols = X_train.select_dtypes(include='bool').columns

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [71]:
X_train = X_train.drop('Open', axis=1)
X_test = X_test.drop('Open', axis=1)

In [72]:
X_train = X_train.drop('Customers', axis=1)
X_test = X_test.drop('Customers', axis=1)

In [73]:
print(X_train.dtypes)

Store                          int64
DayOfWeek                      int32
Promo                          int64
SchoolHoliday                  int64
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                 object
Year                           int32
Month                          int32
Day                            int32
WeekOfYear                     int64
IsWeekend                      int64
IsPromo                        int64
StoreType_b                    int64
StoreType_c                    int64
StoreType_d                    int64
Assortment_b                   int64
Assortment_c                   int64
StateHoliday_0                 int64
StateHoliday_a                 int64
StateHoliday_b                 int64
StateHoliday_c                 int64
dtype: object


In [74]:
X_train = X_train.drop('PromoInterval', axis=1)
X_test = X_test.drop('PromoInterval', axis=1)

In [75]:
print(X_train.dtypes)

Store                          int64
DayOfWeek                      int32
Promo                          int64
SchoolHoliday                  int64
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
Year                           int32
Month                          int32
Day                            int32
WeekOfYear                     int64
IsWeekend                      int64
IsPromo                        int64
StoreType_b                    int64
StoreType_c                    int64
StoreType_d                    int64
Assortment_b                   int64
Assortment_c                   int64
StateHoliday_0                 int64
StateHoliday_a                 int64
StateHoliday_b                 int64
StateHoliday_c                 int64
dtype: object


In [76]:
print(X_train.select_dtypes(include='object').columns)

Index([], dtype='object')


In [77]:
print("Any object columns:", X_train.select_dtypes(include='object').shape[1])
print("Any bool columns:", X_train.select_dtypes(include='bool').shape[1])

Any object columns: 0
Any bool columns: 0


In [79]:
# Save processed datasets
X_train.to_csv('../data/processed/X_train.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)

X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
print("Code Executed Successfully")

Code Executed Successfully
